Colab de referência: https://colab.research.google.com/drive/1s0wK-XDQOdUubutjqhJVQL-6q6jKAfy9

In [46]:
import pandas as pd
from sklearn.model_selection import train_test_split

# carregar dados do dataset titanic
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'
titanic = pd.read_csv(url)
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


#### Holdout
Como fazer?
* Selecionar características relevantes: *age* (idade) e *fare* (tarifa), como tentativa de predizer se o passsageiro sobreviveu;
* Remove dados faltantes: garante que o modelo treinado terá os dados completos;
* Divide os dados: treinamento (60%), validação (20%) e teste (20%)
* Verifica as dimensões: confirma que os conjuntos foram divididos corretamente.

In [23]:
# remover dados faltantes das colunas de interesse (age e idade) e da coluna alvo (survived)
titanic_model = titanic[['age', 'fare', 'survived']].dropna()

# colocar 'age' e 'fare' como features (variáveis explicativas) para o modelo e survived como target (variável a ser prevista)
X = titanic_model[['age', 'fare']]
y = titanic_model['survived']

# 60% para o treinamento / 40% para o temp (que será dividido em 20% teste e 20% validação)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)

# Pegando os 40% de temp e dividindo metade para validação e metade para teste.
# Divisão final: 60% (treinamento), 20% (validação) e 20% (teste).
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [24]:
# Verificando o tamanho dos conjuntos de treinamento, validação e teste
print("Tamanho do dataset:", titanic_model.shape)
print(f"Tamanho do X de treinamento: {X_train.shape}")
print("Tamanho do y de treinamento:", y_train.shape)
print("Tamanho do X de validação:", X_val.shape)
print("Tamanho do y de validação:", y_val.shape)
print("Tamanho do X de teste:", X_test.shape)
print("Tamanho do y de teste", y_test.shape)

Tamanho do dataset: (714, 3)
Tamanho do X de treinamento: (428, 2)
Tamanho do y de treinamento: (428,)
Tamanho do X de validação: (143, 2)
Tamanho do y de validação: (143,)
Tamanho do X de teste: (143, 2)
Tamanho do y de teste (143,)


##### Treinar, validar e testar

In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# noinspection PyShadowingNames
def treina_modelo_usando_regressao_logistica(X_train,
                                             y_train,
                                             X_val,
                                             y_val):
    # carregar o modelo
    model = LogisticRegression()
    # treinar
    model.fit(X_train, y_train)

    # avaliar com o conjunto de validação
    y_val_pred = model.predict(X_val)
    validation_acc = accuracy_score(y_val, y_val_pred)
    validation_acc = validation_acc * 100
    print(f"Acurácia na validação: {validation_acc:.2f}%")

    # avaliar o modelo no conjunto de teste
    y_test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_acc = test_acc * 100
    print(f"Acurácia no teste: {test_acc:.2f}%")


In [32]:
# chamando a função que treina, valida e testa o modelo
treina_modelo_usando_regressao_logistica(X_train, y_train, X_val, y_val)

Acurácia na validação: 64.34%
Acurácia no teste: 60.84%


#### Validação cruzada (***K-Fold***)
Como funciona?

* Divide os dados em k vezes (definimos como **folds**) e comparamos entre eles;
* Estimativa mais robusta, reduzindo possíveis vieses que poderia ocorrer devido a **uma única divsão aletória dos dados** em treinamento e teste;
* Por exemplo, ao repetir o processo de treinamento e validação em 5 grupos (5 **folds**), utilizamos diferentes combinações de dados para treino e teste, e obtemos diferentes valores de acurácia. Este fato proporciona uma visão mais precisa do desempenho geral do modelo.

In [33]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5)

for train_index, val_index in kf.split(X_train):
    X_train_fold, X_val_fold = X.iloc[train_index] , X.iloc[val_index]
    y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

    treina_modelo_usando_regressao_logistica(X_train_fold, y_train_fold, X_val_fold, y_val_fold)

Acurácia na validação: 58.14%
Acurácia no teste: 60.84%
Acurácia na validação: 63.95%
Acurácia no teste: 60.84%
Acurácia na validação: 66.28%
Acurácia no teste: 58.04%
Acurácia na validação: 63.53%
Acurácia no teste: 62.24%
Acurácia na validação: 56.47%
Acurácia no teste: 59.44%


In [48]:
#### Exercício 1 - Divida o conjunto de dados do titanic em 70% treino, 15% validação e 15% teste.
import pandas as pd
from sklearn.model_selection import train_test_split

url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'

titanic = pd.read_csv(url)

# titanic.head()

# divisão

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val_, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Tamanho do dataset: {titanic.shape}")
print(f"Tamanho X de treinamento: {X_train.shape}")
print(f"Tamanho Y de treinamento: {y_train.shape}")
print(f"Tamanho X de validação: {X_val.shape}")
print(f"Tamanho y de validação: {y_val.shape}")
print(f"Tamanho X de teste: {X_test.shape}")
print(f"Tamanho y de teste: {y_test.shape}")

Tamanho do dataset: (891, 15)
Tamanho X de treinamento: (499, 2)
Tamanho Y de treinamento: (499,)
Tamanho X de validação: (107, 2)
Tamanho y de validação: (143,)
Tamanho X de teste: (108, 2)
Tamanho y de teste: (108,)
